## GPU ENGINE. UVM and streaming for Single GPU
Like polars streaming engine, for GPU an streaming memory engine is in beta, aslo taking advantage of UVM (unified virutal memory) combining machine RAM and GPU VRAM.

[UVM, data larger than GPU memory post (NVIDIA blog)](https://developer.nvidia.com/blog/how-to-work-with-data-exceeding-vram-in-the-polars-gpu-engine/)

[UVM, data larger than GPU memory post (Polars blog)](https://pola.rs/posts/uvm-larger-than-ram-gpu/)

[Polars GPU Streaming post (NVIDIA blog)](https://developer.nvidia.com/blog/rapids-adds-gpu-polars-streaming-a-unified-gnn-api-and-zero-code-ml-speedups)

In [1]:
import polars as pl
import os

zone_schema = {"LocationID": pl.Int32, "Borough": pl.String, "Zone": pl.String, "service_zone": pl.String}

df_zones = pl.scan_csv("data/nyc-taxi/taxi_zone_lookup.csv", schema=zone_schema)

df = (
    pl.scan_parquet("data/nyc-taxi/yellow.parquet")
    .join(df_zones, left_on="PULocationID", right_on="LocationID")
    .rename({"Borough": "PU_Borough", "Zone": "PU_Zone", "service_zone": "PU_service_zone"})
    .join(df_zones, left_on="DOLocationID", right_on="LocationID")
    .rename({"Borough": "DO_Borough", "Zone": "DO_Zone", "service_zone": "DO_service_zone"})
)
df_green = (
    pl.scan_parquet("data/nyc-taxi/green.parquet")
    .join(df_zones, left_on="PULocationID", right_on="LocationID")
    .rename({"Borough": "PU_Borough", "Zone": "PU_Zone", "service_zone": "PU_service_zone"})
    .join(df_zones, left_on="DOLocationID", right_on="LocationID")
    .rename({"Borough": "DO_Borough", "Zone": "DO_Zone", "service_zone": "DO_service_zone"})
)

In [2]:
df.head().collect(engine="streaming")

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,year,month,PU_Borough,PU_Zone,PU_service_zone,DO_Borough,DO_Zone,DO_service_zone
i16,datetime[ms],datetime[ms],u16,f32,u8,str,u16,u16,u8,f32,f32,f32,f32,f32,f32,f32,f32,f32,i16,i8,str,str,str,str,str,str
2,2014-01-02 14:25:00,2014-01-02 14:45:00,1,8.83,1,null,236,66,1,27.0,0.0,0.5,5.4,0.0,0.0,32.900002,null,null,2014,1,"""Manhattan""","""Upper East Side North""","""Yellow Zone""","""Brooklyn""","""DUMBO/Vinegar Hill""","""Boro Zone"""
2,2014-01-02 14:53:00,2014-01-02 15:00:00,1,1.22,1,null,225,82,2,6.5,0.0,0.5,0.0,0.0,0.0,7.0,null,null,2014,1,"""Brooklyn""","""Stuyvesant Heights""","""Boro Zone""","""Queens""","""Elmhurst""","""Boro Zone"""
1,2014-01-02 11:20:54,2014-01-02 11:47:03,1,14.5,1,"""N""",141,14,1,41.0,0.0,0.5,6.0,5.33,0.0,52.830002,null,null,2014,1,"""Manhattan""","""Lenox Hill West""","""Yellow Zone""","""Brooklyn""","""Bay Ridge""","""Boro Zone"""
2,2014-01-02 11:03:00,2014-01-02 11:13:00,1,2.76,1,null,261,66,2,10.5,0.0,0.5,0.0,0.0,0.0,11.0,null,null,2014,1,"""Manhattan""","""World Trade Center""","""Yellow Zone""","""Brooklyn""","""DUMBO/Vinegar Hill""","""Boro Zone"""
1,2014-01-02 12:28:02,2014-01-02 12:48:46,2,5.2,1,"""N""",141,82,2,19.5,0.0,0.5,0.0,0.0,0.0,20.0,null,null,2014,1,"""Manhattan""","""Lenox Hill West""","""Yellow Zone""","""Queens""","""Elmhurst""","""Boro Zone"""


## GPU in memory

In [3]:
os.environ["POLARS_GPU_ENABLE_CUDA_MANAGED_MEMORY"] = "0"  # Forcing in-memory engine to not use UVM, just for the demo
gpu_in_mem_engine = pl.GPUEngine(executor="in-memory", raise_on_fail=True)

In [4]:
%%time
df.group_by(["trip_distance", "DO_Borough", "PU_Borough", "total_amount"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough", "total_amount"]).head().collect(
    engine=gpu_in_mem_engine
)  # RuntimeError: parallel_for failed: cudaErrorMemoryAllocation: out of memory

CPU times: user 3.27 s, sys: 614 ms, total: 3.88 s
Wall time: 982 ms


MemoryError: std::bad_alloc: out_of_memory: CUDA error (failed to allocate 16572509184 bytes) at: /tmp/pip-build-env-xlbhj6xg/normal/lib/python3.12/site-packages/librmm/include/rmm/mr/device/cuda_async_view_memory_resource.hpp:102: cudaErrorMemoryAllocation out of memory

In [ ]:
# Recommended to reset kernel to free GPU correctly because some part of the dataset is cached in memory from previous execution
# in the notebook and import again polars import and df definition at the top of the notebook

In [5]:
# This is the manual configuration released in the first UVM post on polars and NVIDIA post.
import rmm

mr = rmm.mr.PrefetchResourceAdaptor(rmm.mr.PoolMemoryResource(rmm.mr.ManagedMemoryResource()))
gpu_streaming_engine = pl.GPUEngine(memory_resource=mr, raise_on_fail=True)


# The new simplified version is passing streaming executor to the GPU engine. This is the default setting if no executor is specified
os.environ["POLARS_GPU_ENABLE_CUDA_MANAGED_MEMORY"] = "1"  # This is the default config
gpu_streaming_engine = pl.GPUEngine(executor="streaming", raise_on_fail=True)

In [6]:
%%time
df.group_by(["trip_distance", "DO_Borough", "PU_Borough"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough"]).head().collect(engine=gpu_streaming_engine)

CPU times: user 8.12 s, sys: 1.79 s, total: 9.91 s
Wall time: 8.5 s


trip_distance,DO_Borough,PU_Borough,trip_count,total_revenue,avg_fare,avg_distance,avg_tip
f32,str,str,u32,f32,f32,f32,f32
-4.0840124e7,"""Queens""","""Queens""",1,2.5,2.5,-4.0840124e7,0.0
-4.0095532e7,"""Manhattan""","""Manhattan""",1,0.0,0.0,-4.0095532e7,1.0
-3.4467e6,"""Queens""","""Manhattan""",1,2.5,2.5,-3.4467e6,0.0
-186318.40625,"""Queens""","""Queens""",1,2.5,2.5,-186318.40625,0.0
-37264.53125,"""Brooklyn""","""Queens""",1,25.809999,25.809999,-37264.53125,0.0


## Multi GPU Streaming. Currently in experimental stage.
#### NOTE: Only run this part in an environment where are multiple GPUs. Kaggle can be a free option to test it.

In [ ]:
# !uv pip install dask_cuda # Dask Cuda client needed

In [ ]:
## Multi GPU Streaming
from dask_cuda import LocalCUDACluster
from dask.distributed import Client

client = Client(LocalCUDACluster())

multi_gpu_streaming = pl.GPUEngine(
    executor="streaming",
    executor_options={"scheduler": "distributed"},  # For a single machine, the executor option is synchronous
)

In [ ]:
%%time
df.group_by(["trip_distance", "DO_Borough", "PU_Borough"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough"]).head().collect(engine=multi_gpu_streaming)